# 3 · Build BioWordVec Vocabulary + Embedding Matrix

**This notebook supersedes `3_BuildGloveVocab.ipynb`.** The proposal's own methodology
(§3.4.1: *"Model BiLSTM akan dikonfigurasi ... menggunakan embedding **BioWordVec** dengan
dimensi **200**"*) specifies BioWordVec, not GloVe — GloVe only appears in §2.1.3 as background
literature-review narrative on the history of word embeddings, never as a stated choice for this
thesis's own model. `MAIN_NB`'s code had drifted from the proposal by using generic GloVe; this
notebook (and the matching patch to `MAIN_NB.ipynb`) brings the implementation back in line with
what §3.4.1 actually commits to in writing.

**Why BioWordVec specifically, beyond "the proposal says so":** BioWordVec
(Zhang, Chen, Yang, Lin, & Lu, 2019 — cited in the proposal's own §2.1.9/references) is trained
on PubMed abstracts + MeSH + (in the extended release used here) MIMIC-III clinical notes, so its
vocabulary actually contains pharmacology/clinical terminology that a Wikipedia/news-trained
embedding like GloVe mostly doesn't. That directly addresses the coverage gap the GloVe version
of this notebook surfaced — a large fraction of DDI corpus vocabulary (drug names, mechanism
terms) fell back to random initialization under GloVe.

**Practical cost you should know about up front:** BioWordVec ships far larger than GloVe.
GloVe 6B is an ~822 MB zip; BioWordVec's smallest usable release is a **13 GB** binary. Loading
that fully into memory with gensim typically needs on the order of 2× the file size in RAM
(~25–30 GB free). If that's not available in your environment, see the "if you can't fit the
full file in memory" note in the config cell below.

## Two release formats — pick one

| File | Size | What it gives you |
|---|---|---|
| `BioWordVec_PubMed_MIMICIII_d200.vec.bin` | 13 GB | Fixed vectors for ~2.3M words, word2vec binary format. Words never seen during BioWordVec's own training still fall back to random init here, same as the GloVe version. |
| `BioWordVec_PubMed_MIMICIII_d200.bin` | 26 GB | The full fastText model. Because fastText composes a word's vector from character n-grams, it can produce a meaningful vector for **any** string, including a drug name it never saw during training — this is the version that actually eliminates the OOV-drug-name problem, not just reduces it. Needs `gensim`'s `load_facebook_vectors` and considerably more RAM (community reports suggest it doesn't comfortably fit on a 16 GB machine). |

Both are hosted by the paper's authors at NCBI:
<https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/> (also mirrored on Figshare via the
[ncbi-nlp/BioWordVec](https://github.com/ncbi-nlp/BioWordVec) repo). This notebook defaults to
the smaller 13 GB `.vec.bin` and treats the fastText `.bin` as an opt-in (`USE_FASTTEXT_SUBWORDS`)
for anyone with the RAM to spare — most of this notebook's logic (tokenizing, vocab assembly,
random-init fallback, saving) is identical either way; only the loading step differs.


In [ ]:
import os
import re
import json
from collections import Counter

import numpy as np
import pandas as pd
from gensim.models import KeyedVectors


## Config

- If you're using the 13 GB `.vec.bin` (default): set `BIOWORDVEC_PATH` and leave
  `USE_FASTTEXT_SUBWORDS = False`.
- If you're using the 26 GB fastText `.bin` and have the RAM for it: set
  `USE_FASTTEXT_SUBWORDS = True` and point `FASTTEXT_MODEL_PATH` at it instead.
- **If you can't fit either file in memory:** the word2vec binary format can be streamed and
  filtered line-by-line in principle, but gensim's own binary-format reader loads the whole
  vocabulary before you can filter it — there's no supported partial-load path for `.bin`
  (unlike GloVe's plain-text `.txt`, which the previous notebook could stream). In that case,
  either run this notebook once on a machine with enough RAM and copy just the two small output
  files (`vocab.json` + `embed_matrix.npy`, typically a few hundred MB at most) back to your
  working environment, or fall back to `3_BuildGloveVocab.ipynb` and document the substitution
  in your methodology chapter (see the discussion of this trade-off in the previous review).


In [ ]:
DATA_DIR    = "data"
BIOWORDVEC_DIM = 200   # must match BiLSTMConfig.embed_dim in MAIN_NB (updated to 200 for BioWordVec)
MIN_FREQ    = 1        # drop corpus words rarer than this (1 = keep everything)
INCLUDE_TEST = True    # build vocab from all 6 splits, not just train/val — see design note
                        # in 3_BuildGloveVocab.ipynb §0 for the rationale (still applies: this
                        # is an external pretrained resource, not something fit to the data,
                        # so including test text is not label leakage).
RANDOM_SEED = 42

# ── Pick ONE embedding source ────────────────────────────────────────────────
USE_FASTTEXT_SUBWORDS = False   # True = use the 26GB fastText .bin (subword OOV composition)
BIOWORDVEC_PATH        = "BioWordVec_PubMed_MIMICIII_d200.vec.bin"   # 13GB word2vec-binary release
FASTTEXT_MODEL_PATH    = "BioWordVec_PubMed_MIMICIII_d200.bin"       # 26GB full fastText model

TEXT_SOURCES = [
    ("holistic_train", f"{DATA_DIR}/holistic_train.csv"),
    ("holistic_val",   f"{DATA_DIR}/holistic_val.csv"),
    ("atomic_train",   f"{DATA_DIR}/atomic_train.csv"),
    ("atomic_val",     f"{DATA_DIR}/atomic_val.csv"),
]
if INCLUDE_TEST:
    TEXT_SOURCES += [
        ("holistic_test", f"{DATA_DIR}/holistic_test.csv"),
        ("atomic_test",   f"{DATA_DIR}/atomic_test.csv"),
    ]

embedding_path = FASTTEXT_MODEL_PATH if USE_FASTTEXT_SUBWORDS else BIOWORDVEC_PATH
if not os.path.exists(embedding_path):
    raise FileNotFoundError(
        f"'{embedding_path}' not found. Download it from "
        f"https://ftp.ncbi.nlm.nih.gov/pub/lu/Suppl/BioSentVec/ "
        f"(see the format comparison table above to pick the right file for "
        f"USE_FASTTEXT_SUBWORDS={USE_FASTTEXT_SUBWORDS})."
    )

missing = [path for _, path in TEXT_SOURCES if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(
        f"Missing input CSV(s): {missing}. Run 0_ExtractData.ipynb, 1_AugmentResponses.ipynb, "
        f"and 2_Atomisation.ipynb first."
    )

print(f"Embedding source : {embedding_path}  (dim={BIOWORDVEC_DIM}, "
      f"fastText subwords={USE_FASTTEXT_SUBWORDS})")
print(f"Text sources     : {[name for name, _ in TEXT_SOURCES]}")


## Tokenize the corpus, count word frequencies

Identical to `3_BuildGloveVocab.ipynb` and to `MAIN_NB.tokenize()` — same regex, same lowercase
rule. BioWordVec's own vocabulary is entirely lowercase by construction (per its documentation),
which matches this pipeline's tokenizer already.


In [ ]:
def simple_tokenize(text):
    return re.findall(r"[a-z0-9']+", str(text).lower())


word_counts = Counter()

for name, path in TEXT_SOURCES:
    df = pd.read_csv(path)
    n_tokens = 0
    for col in ("premise", "hypothesis"):
        if col not in df.columns:
            continue
        for text in df[col].dropna():
            toks = simple_tokenize(text)
            word_counts.update(toks)
            n_tokens += len(toks)
    print(f"{name:16s}: {len(df):>6,} rows -> {n_tokens:>9,} tokens")

print(f"\nTotal unique corpus words (pre-frequency-filter): {len(word_counts):,}")

corpus_vocab = [w for w, c in word_counts.items() if c >= MIN_FREQ]
corpus_vocab.sort(key=lambda w: -word_counts[w])
print(f"Corpus words kept at MIN_FREQ={MIN_FREQ}: {len(corpus_vocab):,}")


## Load BioWordVec and look up the corpus vocabulary

Two branches, selected by `USE_FASTTEXT_SUBWORDS`:

- **Word2vec-binary branch (default):** loads the full ~2.3M-word BioWordVec table with gensim,
  then keeps only the vectors for words that appear in your corpus. This step is the one that
  needs the ~25–30 GB of free RAM mentioned above — it happens once, right here.
- **FastText branch (opt-in):** loads the fastText model and queries every corpus word directly.
  There is no "not found" case in this branch for ordinary text — fastText composes a vector
  from subword n-grams even for words absent from BioWordVec's own training vocabulary, which is
  exactly the property that matters for drug names.


In [ ]:
biowordvec_vectors = {}

if USE_FASTTEXT_SUBWORDS:
    from gensim.models.fasttext import load_facebook_vectors
    print("Loading fastText model (this can take a while and a lot of RAM)...")
    ft = load_facebook_vectors(FASTTEXT_MODEL_PATH)
    for w in corpus_vocab:
        try:
            biowordvec_vectors[w] = np.asarray(ft[w], dtype=np.float32)
        except Exception:
            pass   # pathological inputs only (e.g. empty string) - real words always compose
else:
    print("Loading BioWordVec word2vec-binary table (this can take a while and a lot of RAM)...")
    kv = KeyedVectors.load_word2vec_format(BIOWORDVEC_PATH, binary=True)
    corpus_vocab_set = set(corpus_vocab)
    for w in corpus_vocab_set:
        if w in kv:
            biowordvec_vectors[w] = np.asarray(kv[w], dtype=np.float32)
    del kv   # free the full ~2.3M-word table now that we've extracted what we need

print(f"\nCorpus words found in BioWordVec : {len(biowordvec_vectors):,} / {len(corpus_vocab):,} "
      f"({100*len(biowordvec_vectors)/max(len(corpus_vocab),1):.1f}%)")

oov_words = [w for w in corpus_vocab if w not in biowordvec_vectors]
print(f"Corpus words NOT covered          : {len(oov_words):,} (random-initialized)")
if oov_words:
    print("\nSample uncovered words (by frequency) - expect this list to be much shorter and "
          "less drug-name-heavy than the GloVe version:")
    for w in sorted(oov_words, key=lambda w: -word_counts[w])[:30]:
        print(f"  {w:20s} freq={word_counts[w]}")


## Assemble `word2idx` and the embedding matrix

Same scheme as the GloVe notebook — `<PAD>=0` (zero vector), `<UNK>=1` (one random vector,
reserved for genuinely unseen words at inference time), then corpus vocabulary in frequency
order. Any corpus word BioWordVec doesn't cover (only relevant in the word2vec-binary branch —
the fastText branch should leave this list nearly empty) gets its **own** independent random
vector rather than sharing `<UNK>`, for the same reason as before: collapsing distinct drug names
into one indistinguishable token would destroy the signal the task depends on most.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

all_vecs = np.stack(list(biowordvec_vectors.values())) if biowordvec_vectors else np.zeros((1, BIOWORDVEC_DIM))
emb_mean, emb_std = all_vecs.mean(axis=0), all_vecs.std(axis=0)

def random_vector():
    return rng.normal(loc=emb_mean, scale=emb_std).astype(np.float32)


word2idx = {"<PAD>": 0, "<UNK>": 1}
vectors  = [np.zeros(BIOWORDVEC_DIM, dtype=np.float32), random_vector()]

for word in corpus_vocab:
    word2idx[word] = len(vectors)
    vectors.append(biowordvec_vectors[word] if word in biowordvec_vectors else random_vector())

embed_matrix = np.stack(vectors).astype(np.float32)

print(f"Final vocab size : {len(word2idx):,}")
print(f"Embedding matrix  : {embed_matrix.shape}")
assert embed_matrix.shape == (len(word2idx), BIOWORDVEC_DIM)
assert word2idx["<PAD>"] == 0 and word2idx["<UNK>"] == 1
assert np.allclose(embed_matrix[0], 0.0)


## Save `data/vocab.json` and `data/embed_matrix.npy`

Same output files, same interface `MAIN_NB`'s `BioWordVecVocab` reads — only the *content* of
the embedding matrix changed (BioWordVec vectors instead of GloVe vectors, 200 dims instead of
100). No other notebook in the pipeline needs to change because of that; only `MAIN_NB`'s
`BiLSTMConfig.embed_dim` needs to move from `100` to `200` to match (already patched — see that
notebook's changelog).


In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

with open(f"{DATA_DIR}/vocab.json", "w") as f:
    json.dump(word2idx, f)

np.save(f"{DATA_DIR}/embed_matrix.npy", embed_matrix)

print(f"Saved {DATA_DIR}/vocab.json         ({len(word2idx):,} entries)")
print(f"Saved {DATA_DIR}/embed_matrix.npy   {embed_matrix.shape}")
print(f"\nBioWordVec coverage: {100*len(biowordvec_vectors)/max(len(corpus_vocab),1):.1f}% of "
      f"corpus vocabulary matched a pretrained vector.")


## Changelog

| # | Issue | Fix |
|---|---|---|
| 1 | `MAIN_NB` used generic GloVe; the proposal's §3.4.1 specifies BioWordVec (dim 200), trained on PubMed/MeSH/MIMIC-III specifically so pharmacology vocabulary isn't dominated by random-initialized OOV rows | This notebook loads real BioWordVec vectors instead of GloVe |
| 2 | BioWordVec ships far larger than GloVe (13–26 GB vs. ~822 MB) with no partial-load path for the binary formats | Documented the RAM/disk trade-off explicitly; defaults to the smaller 13 GB release, with the 26 GB fastText release available as an opt-in for full subword OOV coverage |
| 3 | The word2vec-binary release still leaves some domain words uncovered (fixed vocabulary, no subword composition) | `USE_FASTTEXT_SUBWORDS=True` switches to the fastText release, which composes a vector for any string — the more complete fix, at a higher RAM cost |
